# CSMS AI - Single Category End-to-End Test
**Category 8.1 - Sertifikasi Sistem Pengelolaan K3LL**

Runs the full pipeline for ONE category with verbose output at every stage.

## Pipeline Stages
1. Setup & VLLM connection check
2. Load Knowledge Base
3. Discover submission folder
4. Extract documents
5. Incremental chunking & embedding (hash-cached SQLite)
6. Multi-query hybrid retrieval
7. Validation (Exact / Fuzzy / LLM)
8. Shared fact sheet
9. Criterion scoring
10. VLLM Juror + Consensus (with hard score gates)
11. Final result & audit report

## 1. Setup & Configuration

In [ ]:
import os, sys, json, re, sqlite3, struct, hashlib, time, difflib
from typing import Dict, Any, List, Optional
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

sys.path.append(os.path.abspath('..'))

# ── Paths ─────────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('..')
MARINE_DIR   = os.path.join(PROJECT_ROOT, 'test_submissions', 'marine')
KB_DIR       = os.path.join(PROJECT_ROOT, 'knowledge_base', 'marine')
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'notebook_v2', 'outputs')
DB_DIR       = os.path.join(OUTPUT_DIR, 'databases')
os.makedirs(DB_DIR, exist_ok=True)

# ── Target Category ───────────────────────────────────────────────────────
TARGET_CATEGORY_ID = '8.1'

# ── Load model config ─────────────────────────────────────────────────────
with open(os.path.join(PROJECT_ROOT, 'configs', 'model_config.json'), 'r') as f:
    model_cfg = json.load(f)
vllm_cfg    = model_cfg['providers'].get('vllm', {})
base_url    = vllm_cfg.get('base_url', 'http://localhost:8000/')
VLLM_API_URL = (base_url.rstrip('/') + '/v1')
VLLM_MODEL   = vllm_cfg.get('model', 'qwen35')

OPENAI_API_KEY     = os.getenv('OPENAI_API_KEY')
EMBEDDING_MODEL    = 'text-embedding-3-small'
EMBEDDING_DIM      = 1536
CHUNK_SIZE         = 800
CHUNK_OVERLAP      = 100
TOP_K              = 20
CONFIDENCE_THRESHOLD = 0.8
FORCE_REINDEX      = False
VERBOSE            = True

IGNORE_FILES = {'expected_result.json', 'readme.md', '.keep', '.placeholder', 'thumbs.db', '.ds_store'}

print(f'Project root : {PROJECT_ROOT}')
print(f'Marine dir   : {MARINE_DIR}')
print(f'VLLM URL     : {VLLM_API_URL}')
print(f'VLLM Model   : {VLLM_MODEL}')
print(f'Emb Model    : {EMBEDDING_MODEL}')
print(f'Target cat   : {TARGET_CATEGORY_ID}')
print(f'Force reindex: {FORCE_REINDEX}')

## 2. VLLM & Embedding Connection Check

In [ ]:
from openai import OpenAI

vllm_client   = OpenAI(api_key='EMPTY', base_url=VLLM_API_URL)
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# VLLM check
print(f'Testing VLLM at {VLLM_API_URL} ...')
try:
    models = [m.id for m in vllm_client.models.list().data]
    print(f'  Available models: {models}')
    if VLLM_MODEL in models:
        print(f'  OK  Model "{VLLM_MODEL}" confirmed.')
    else:
        print(f'  WARN Model "{VLLM_MODEL}" NOT in list. Update VLLM_MODEL.')
except Exception as e:
    print(f'  FAIL VLLM connection error: {e}')

# Embedding check
if openai_client:
    try:
        r = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=['test'])
        print(f'  OK  OpenAI Embedding dim={len(r.data[0].embedding)}')
    except Exception as e:
        print(f'  FAIL OpenAI Embedding: {e}')
else:
    print('  WARN OPENAI_API_KEY not set -- zero vectors will be used.')

## 3. Load Knowledge Base

In [ ]:
def load_kb(category_id: str) -> Dict[str, Any]:
    clean_id = re.match(r'^[0-9]+(\.[0-9]+)?', str(category_id).strip())
    clean_id = clean_id.group(0) if clean_id else str(category_id).strip()
    result = {}
    for kind, sub, prefix in [
        ('category',             'categories',            ''),
        ('scoring_rules',        'scoring_rules',         'SR-'),
        ('validation_rules',     'validation_rules',      'VR-'),
        ('document_requirements','document_requirements', 'DR-'),
        ('traceability',         'traceability',          'TR-'),
    ]:
        fpath = os.path.join(KB_DIR, sub, f'{prefix}{clean_id}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                result[kind] = json.load(f)
        else:
            result[kind] = {}
    return result

REGULATORY_CONTEXT = {
    '1': 'Kepemimpinan dan Komitmen - ISO 45001 Clause 5.1, ISM Code Element 1.4',
    '2': 'Kebijakan dan Sasaran Strategis K3LL - ISO 45001 Clause 5.2',
    '3': 'Organisasi, Tanggung Jawab, Sumber Daya - STCW, ISM Code Element 3',
    '4': 'Penanganan Risiko - ISO 31000, JSA, HAZID/HAZOP, AKHLAK Safety Culture',
    '5': 'Perencanaan dan Prosedur - SOLAS Ch.III/IV, MARPOL Annex I-VI',
    '6': 'Implementasi dan Pemantauan Kinerja - ISM Code Element 9/10',
    '7': 'Audit dan Tinjauan - Management Review, ISM Code Element 12',
    '8': 'Ciri-ciri Tambahan - ISPS Code, Environmental Leadership, CSR'
}

def expand_kb_context(cat_id: str, kb: Dict[str, Any]) -> Dict[str, Any]:
    parent = cat_id.split('.')[0]
    kb['regulatory_context'] = REGULATORY_CONTEXT.get(parent, 'Standard Marine Safety Context')
    return kb

kb_data = load_kb(TARGET_CATEGORY_ID)
kb_data = expand_kb_context(TARGET_CATEGORY_ID, kb_data)

cat = kb_data.get('category', {})
sr  = kb_data.get('scoring_rules', {})
vr  = kb_data.get('validation_rules', {})

print(f'Category name : {cat.get("category_name","?")}')
print(f'SR rule id    : {sr.get("rule_id","?")}')
print(f'Reg. context  : {kb_data.get("regulatory_context","")}')
print()
print('--- Score Levels ---')
for level, desc in sr.get('score_levels', {}).items():
    print(f'  [{level}] {str(desc)[:120]}...')
print()
print('--- Evaluation Criteria ---')
print(sr.get('evaluation_criteria', 'N/A'))
print()
print('--- Mandatory Fields ---')
print(sr.get('mandatory_fields', 'N/A'))
print()
print('--- Reasoning Hint ---')
print(sr.get('reasoning_hint', 'N/A'))

## 4. Discover Category Submission Folder

In [ ]:
def find_category_folder(marine_dir: str, cat_id: str) -> Optional[str]:
    pattern = re.compile(r'^' + re.escape(cat_id) + r'(\s|$)')
    for item in sorted(os.listdir(marine_dir)):
        full = os.path.join(marine_dir, item)
        if os.path.isdir(full) and pattern.match(item):
            return full
    return None

cat_folder = find_category_folder(MARINE_DIR, TARGET_CATEGORY_ID)
if not cat_folder:
    raise FileNotFoundError(f'No folder for category {TARGET_CATEGORY_ID} in {MARINE_DIR}')

expected_score = None
exp_file = os.path.join(cat_folder, 'expected_result.json')
if os.path.exists(exp_file):
    with open(exp_file, 'r', encoding='utf-8') as f:
        expected_score = json.load(f).get('expected_score')

print(f'Folder  : {cat_folder}')
print(f'Expected: {expected_score}')
print()
print('Files in submission folder:')
all_files = []
for root, _, files in os.walk(cat_folder):
    for fn in sorted(files):
        if fn.lower() not in IGNORE_FILES:
            rel  = os.path.relpath(os.path.join(root, fn), cat_folder)
            size = os.path.getsize(os.path.join(root, fn))
            all_files.append({'file': rel, 'size_kb': round(size/1024, 1)})
            print(f'  {rel:<70} ({size//1024} KB)')
print(f'Total: {len(all_files)} files')

## 5. Document Extraction

In [ ]:
try:
    from engine.extractors.router import ExtractorRouter
    from engine.aggregators.document_classifier import DocumentClassifier
    from engine.aggregators.priority_assigner import PriorityAssigner
    extractor  = ExtractorRouter()
    classifier = DocumentClassifier()
    assigner   = PriorityAssigner()
    ENGINE_AVAILABLE = True
    print('Engine extractors loaded.')
except Exception as e:
    print(f'Engine unavailable ({e}) -- using text fallback.')
    ENGINE_AVAILABLE = False

def extract_documents(folder_path: str) -> List[Dict[str, Any]]:
    extracted = []
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):
            if file.lower() in IGNORE_FILES:
                continue
            fpath    = os.path.join(root, file)
            rel_path = os.path.relpath(fpath, folder_path).replace(os.sep, '/')
            try:
                if ENGINE_AVAILABLE:
                    doc_data = extractor.extract_document(fpath)
                    if doc_data.get('status') != 'error':
                        doc_data['file'] = rel_path
                        cls = classifier.classify(fpath, doc_data.get('metadata', {}))
                        doc_data['classification'] = cls
                        doc_data['priority'] = assigner.assign_priority(cls)
                        extracted.append(doc_data)
                    else:
                        print(f'  WARN engine error for {file}: {doc_data.get("error","")}')
                else:
                    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                        txt = f.read()
                    extracted.append({'file': rel_path, 'content': txt,
                                      'metadata': {}, 'classification': 'Unclassified', 'priority': 4})
            except Exception as e:
                print(f'  FAIL extraction for {file}: {e}')
    return extracted

print(f'Extracting from {cat_folder} ...')
t0   = time.time()
docs = extract_documents(cat_folder)
print(f'Extracted {len(docs)} documents in {time.time()-t0:.1f}s')
print()
for d in docs:
    clen = len(d.get('content','') or ' '.join(str(x) for x in d.get('evidence',[])))
    print(f'  OK  {d["file"]:<70} {clen:>6} chars  pri={d.get("priority","?")}')

## 6. Chunking & Incremental Embedding (Hash-Cached SQLite)

In [ ]:
def get_doc_hash(file_path: str) -> str:
    prefix = '\\\\' + '\\\\?\\\\'
    abs_p  = os.path.abspath(file_path)
    if os.name == 'nt' and not abs_p.startswith(prefix):
        abs_p = prefix + abs_p
    try:
        with open(abs_p, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except Exception as e:
        print(f'  Hash error for {file_path}: {e}')
        return ''

def emb_to_blob(emb):
    return struct.pack(f'{len(emb)}f', *emb)

def blob_to_emb(blob):
    if not blob: return [0.0]*EMBEDDING_DIM
    n = len(blob)//4
    return list(struct.unpack(f'{n}f', blob))

def embed_texts(texts: List[str]) -> List[List[float]]:
    if not texts: return []
    if not openai_client: return [[0.0]*EMBEDDING_DIM for _ in texts]
    result = []
    for i in range(0, len(texts), 100):
        batch = texts[i:i+100]
        try:
            resp = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
            result.extend([x.embedding for x in resp.data])
        except Exception as e:
            print(f'  Embedding error: {e}')
            result.extend([[0.0]*EMBEDDING_DIM]*len(batch))
    return result

def init_db(db_path: str, force: bool = False) -> sqlite3.Connection:
    if force and os.path.exists(db_path):
        os.remove(db_path)
        print('  Force rebuild: removed old DB.')
    conn = sqlite3.connect(db_path)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('''CREATE TABLE IF NOT EXISTS chunks (
        chunk_id TEXT PRIMARY KEY, category_id TEXT NOT NULL,
        filename TEXT NOT NULL, doc_type TEXT, tier TEXT, priority INTEGER,
        evidence_idx INTEGER, chunk_idx INTEGER, text TEXT NOT NULL,
        char_count INTEGER, embedding BLOB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.execute('''CREATE TABLE IF NOT EXISTS doc_metadata (
        filepath TEXT PRIMARY KEY, category_id TEXT,
        doc_hash TEXT, indexed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_cat ON chunks(category_id)')
    conn.execute('''CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5(
        chunk_id, text, filename, category_id,
        content=\'chunks\', content_rowid=\'rowid\')''')
    conn.execute('DROP TRIGGER IF EXISTS chunks_ai')
    conn.execute('''CREATE TRIGGER chunks_ai AFTER INSERT ON chunks BEGIN
        INSERT INTO chunks_fts(chunk_id,text,filename,category_id)
        VALUES(NEW.chunk_id,NEW.text,NEW.filename,NEW.category_id); END''')
    conn.commit()
    return conn

def chunk_doc(doc: Dict) -> List[Dict]:
    parts = []
    if doc.get('content'):
        raw = doc['content']
        start = 0
        while start < len(raw):
            parts.append(raw[start:start+CHUNK_SIZE])
            start += CHUNK_SIZE - CHUNK_OVERLAP
    for ev in doc.get('evidence', []):
        ev = str(ev)
        start = 0
        while start < len(ev):
            parts.append(ev[start:start+CHUNK_SIZE])
            start += CHUNK_SIZE - CHUNK_OVERLAP
    chunks = []
    for idx, part in enumerate(parts):
        if not part.strip(): continue
        cid = hashlib.md5(f"{doc.get('category_id','')}_{doc['file']}_{idx}_{part[:30]}".encode()).hexdigest()
        chunks.append({'chunk_id': cid, 'category_id': doc.get('category_id',''),
                        'filename': doc['file'], 'doc_type': doc.get('doc_type','doc'),
                        'tier': doc.get('tier','Unclassified'), 'priority': doc.get('priority',4),
                        'evidence_idx': idx, 'chunk_idx': 0,
                        'text': part, 'char_count': len(part)})
    return chunks

def store_chunks(conn, chunks):
    cur = conn.cursor()
    n = 0
    for ch in chunks:
        blob = emb_to_blob(ch['embedding']) if ch.get('embedding') else None
        try:
            cur.execute(
                'INSERT OR REPLACE INTO chunks VALUES(?,?,?,?,?,?,?,?,?,?,?,CURRENT_TIMESTAMP)',
                (ch['chunk_id'], ch['category_id'], ch['filename'], ch.get('doc_type','doc'),
                 ch.get('tier','Unclassified'), ch.get('priority',4), ch.get('evidence_idx',0),
                 ch.get('chunk_idx',0), ch['text'], ch['char_count'], blob))
            n += 1
        except Exception as e:
            print(f'  DB insert error: {e}')
    conn.commit()
    return n

# ── Run ───────────────────────────────────────────────────────────────────
db_path = os.path.join(DB_DIR, f'evidence_{TARGET_CATEGORY_ID}.db')
conn    = init_db(db_path, force=FORCE_REINDEX)
cur     = conn.cursor()

to_index, skipped = [], 0
for d in docs:
    d['category_id'] = TARGET_CATEGORY_ID
    full = os.path.join(cat_folder, d['file'].replace('/', os.sep))
    d['doc_hash'] = get_doc_hash(full)
    cur.execute('SELECT doc_hash FROM doc_metadata WHERE filepath=?', (d['file'],))
    row = cur.fetchone()
    if FORCE_REINDEX or row is None or row[0] != d['doc_hash']:
        to_index.append(d)
    else:
        skipped += 1

print(f'Hash check: {len(to_index)} to index, {skipped} cached (skipping).')

if to_index:
    t0 = time.time()
    all_chunks = []
    for d in to_index:
        all_chunks.extend(chunk_doc(d))
    print(f'  Chunked into {len(all_chunks)} chunks.')
    print(f'  Embedding {len(all_chunks)} chunks...')
    embs = embed_texts([c['text'] for c in all_chunks])
    for i, e in enumerate(embs): all_chunks[i]['embedding'] = e
    n = store_chunks(conn, all_chunks)
    for d in to_index:
        cur.execute('INSERT OR REPLACE INTO doc_metadata VALUES(?,?,?,CURRENT_TIMESTAMP)',
                    (d['file'], TARGET_CATEGORY_ID, d['doc_hash']))
    conn.commit()
    print(f'  Indexed {n} chunks in {time.time()-t0:.1f}s.')
else:
    cur.execute('SELECT COUNT(*) FROM chunks WHERE category_id=?', (TARGET_CATEGORY_ID,))
    print(f'  Using {cur.fetchone()[0]} cached chunks from DB.')

## 7. Multi-Query Hybrid Retrieval

In [ ]:
def build_queries(kb: Dict) -> List[str]:
    ec = kb.get('scoring_rules', {}).get('evaluation_criteria', '')
    qs = [l.strip() for l in ec.split('\n') if re.match(r'^\[\d+\]', l.strip())]
    if not qs:
        cat = kb.get('category', {})
        qs  = [cat.get('category_name', 'CSMS compliance evidence')]
    return qs

def retrieve(conn, query: str, cat_id: str, top_k: int = 5) -> List[Dict]:
    cur = conn.cursor()
    cur.execute('SELECT chunk_id,filename,text,tier,priority,embedding FROM chunks WHERE category_id=?', (cat_id,))
    rows = cur.fetchall()
    if not rows:
        cur.execute('SELECT chunk_id,filename,text,tier,priority,embedding FROM chunks')
        rows = cur.fetchall()
    if not rows: return []
    q_vec  = np.array(embed_texts([query])[0], dtype=np.float32)
    q_norm = np.linalg.norm(q_vec)
    hits = []
    for cid, fn, txt, tier, pri, blob in rows:
        c_vec  = np.array(blob_to_emb(blob), dtype=np.float32)
        c_norm = np.linalg.norm(c_vec)
        sim    = float(np.dot(q_vec, c_vec)/(q_norm*c_norm)) if q_norm>0 and c_norm>0 else 0.0
        hits.append({'chunk_id':cid,'filename':fn,'text':txt,'tier':tier,'priority':pri,'similarity':sim})
    hits.sort(key=lambda x: x['similarity'], reverse=True)
    return hits[:top_k]

queries = build_queries(kb_data)
print(f'Generated {len(queries)} queries:')
for i, q in enumerate(queries):
    print(f'  [{i+1}] {q[:110]}')

seen, retrieved_all = set(), []
t0 = time.time()
for q in queries:
    for r in retrieve(conn, q, TARGET_CATEGORY_ID, top_k=5):
        if r['chunk_id'] not in seen:
            seen.add(r['chunk_id'])
            retrieved_all.append(r)
retrieved = retrieved_all[:TOP_K]
print(f'\nRetrieved {len(retrieved)} unique chunks in {time.time()-t0:.1f}s.')
print('\nTop 5 chunks:')
for i, r in enumerate(retrieved[:5]):
    snippet = r['text'][:120].replace('\n',' ')
    print(f'  [{i+1}] sim={r["similarity"]:.3f} | {r["filename"]}')
    print(f'       {snippet}...')

## 8. Validation - Exact / Fuzzy / LLM Mandatory Field Matching

In [ ]:
def normalize_field(field: str) -> str:
    return re.sub(r'^[-*\s]+', '', field).strip().lower()

def fuzzy_match(keyword: str, text: str, threshold: float = 0.85) -> bool:
    if not keyword: return False
    words = text.split()
    kw_words = keyword.split()
    if len(kw_words) == 1:
        return any(difflib.SequenceMatcher(None, keyword, w).ratio() >= threshold for w in words)
    n = len(kw_words)
    for i in range(len(words)-n+1):
        if difflib.SequenceMatcher(None, keyword, ' '.join(words[i:i+n])).ratio() >= threshold:
            return True
    return False

def llm_extract_fields(missing: List[str], text_sample: str) -> List[str]:
    if not missing: return []
    prompt = (
        'Given the document text, check which listed fields are conceptually present '
        '(even if phrased differently). Return ONLY a JSON array of field strings found.\n\n'
        f'Document text (4000 chars):\n{text_sample[:4000]}\n\n'
        f'Fields to check:\n{json.dumps(missing, ensure_ascii=False)}'
    )
    try:
        resp = vllm_client.chat.completions.create(
            model=VLLM_MODEL,
            messages=[{'role':'user','content':prompt}],
            temperature=0.0, max_tokens=512,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```[a-z]*\n?','',raw).rstrip('`').strip()
        found = json.loads(raw)
        return [f for f in missing if isinstance(found,list) and f in found]
    except Exception as e:
        print(f'  LLM field extraction warning: {e}')
        return []

def run_validation(kb: Dict, docs: List[Dict]) -> Dict:
    vr  = kb.get('validation_rules', {})
    sr  = kb.get('scoring_rules', {})
    kws = vr.get('keywords') or vr.get('ai_checkpoint_keywords') or []
    mf_raw = sr.get('mandatory_fields','') or vr.get('mandatory_fields','')
    if isinstance(mf_raw, list):
        mfs = mf_raw
    else:
        mfs = [l.strip() for l in str(mf_raw).split('\n') if l.strip()]
    all_text = ' '.join(
        d.get('content','') or ' '.join(str(x) for x in d.get('evidence',[]))
        for d in docs
    ).lower()
    matched_kw = [kw for kw in kws if kw.lower() in all_text]
    missing_kw = [kw for kw in kws if kw not in matched_kw]
    exact  = [mf for mf in mfs if normalize_field(mf) in all_text]
    miss1  = [mf for mf in mfs if mf not in exact]
    fuzzy  = [mf for mf in miss1 if fuzzy_match(normalize_field(mf), all_text)]
    miss2  = [mf for mf in miss1 if mf not in fuzzy]
    via_llm= llm_extract_fields(miss2, all_text)
    miss3  = [mf for mf in miss2 if mf not in via_llm]
    all_matched = exact + fuzzy + via_llm
    kw_score = len(matched_kw)/len(kws) if kws else 1.0
    mf_score = len(all_matched)/len(mfs) if mfs else 1.0
    val_conf = round(0.4*float(len(docs)>0) + 0.3*kw_score + 0.3*mf_score, 3)
    return {
        'has_documents': len(docs)>0, 'doc_count': len(docs),
        'matched_keywords': matched_kw, 'missing_keywords': missing_kw,
        'matched_mandatory_fields': all_matched, 'matched_exact': exact,
        'matched_fuzzy': fuzzy, 'matched_llm': via_llm,
        'missing_mandatory_fields': miss3,
        'keyword_score': round(kw_score,3), 'field_score': round(mf_score,3),
        'confidence': val_conf,
    }

print('Running validation...')
t0 = time.time()
validation = run_validation(kb_data, docs)
print(f'Validation complete in {time.time()-t0:.1f}s.')
print()
print(f'  Docs found          : {validation["doc_count"]}')
print(f'  Keyword match       : {len(validation["matched_keywords"])}/{len(validation["matched_keywords"])+len(validation["missing_keywords"])} = {validation["keyword_score"]*100:.0f}%')
print(f'  Mandatory fields    : {len(validation["matched_mandatory_fields"])}/{len(validation["matched_mandatory_fields"])+len(validation["missing_mandatory_fields"])} = {validation["field_score"]*100:.0f}%')
print(f'    via exact         : {len(validation["matched_exact"])}')
print(f'    via fuzzy         : {len(validation["matched_fuzzy"])}')
print(f'    via LLM           : {len(validation["matched_llm"])}')
print(f'  Validation conf     : {validation["confidence"]}')
print()
print(f'  Matched keywords    : {validation["matched_keywords"][:8]}')
print(f'  Missing keywords    : {validation["missing_keywords"][:8]}')
print(f'  Matched mand.fields : {validation["matched_mandatory_fields"][:5]}')
print(f'  Missing mand.fields : {validation["missing_mandatory_fields"][:5]}')

## 9. Shared Fact Sheet

In [ ]:
fact_sheet = {
    'category_id':             TARGET_CATEGORY_ID,
    'category_name':           cat.get('category_name', f'Category {TARGET_CATEGORY_ID}'),
    'regulatory_context':      kb_data.get('regulatory_context',''),
    'evaluation_criteria':     sr.get('evaluation_criteria',''),
    'has_documents':           validation['has_documents'],
    'doc_count':               validation['doc_count'],
    'matched_keywords':        validation['matched_keywords'],
    'missing_keywords':        validation['missing_keywords'],
    'matched_mandatory_fields': validation['matched_mandatory_fields'],
    'missing_mandatory_fields': validation['missing_mandatory_fields'],
    'evidence_chunks_count':   len(retrieved),
    'top_evidence':            [{'file':c['filename'],'tier':c['tier'],'snippet':c['text'][:200].replace('\n',' ')} for c in retrieved[:10]],
    'validation_confidence':   validation['confidence'],
    'generated_at':            datetime.now().isoformat(),
}

print('Fact Sheet:')
print(f'  Category   : {fact_sheet["category_name"]}')
print(f'  Has docs   : {fact_sheet["has_documents"]} ({fact_sheet["doc_count"]} files)')
print(f'  Chunks     : {fact_sheet["evidence_chunks_count"]}')
print(f'  Val conf   : {fact_sheet["validation_confidence"]}')
print('  Top 3 evidence:')
for i, e in enumerate(fact_sheet['top_evidence'][:3]):
    print(f'    [{i+1}] {e["file"]} | {e["snippet"][:100]}...')

## 10. Criterion-Level Scoring (Pre-LLM Baseline)

In [ ]:
def snap(raw: float) -> int:
    if raw <= 0.0: return 0
    if raw <= 3.0: return 3
    if raw <= 6.0: return 6
    return 10

def evaluate_criteria(fs: Dict, kb: Dict) -> Dict:
    if not fs['has_documents']:
        return {'raw_score':0.0,'final_score':0,'confidence':1.0,'review_flag':False,
                'rationale':'No submission documents.'  }
    tot_kw = len(fs['matched_keywords']) + len(fs['missing_keywords'])
    tot_mf = len(fs['matched_mandatory_fields']) + len(fs['missing_mandatory_fields'])
    kw_s   = len(fs['matched_keywords'])/tot_kw if tot_kw > 0 else 1.0
    mf_s   = len(fs['matched_mandatory_fields'])/tot_mf if tot_mf > 0 else 1.0
    raw    = min(10.0, max(0.0, 4.0*kw_s + 4.0*mf_s + (2.0 if fs['doc_count']>=2 else 1.0)))
    final  = snap(raw)
    conf   = round(min(1.0, 0.5*fs['validation_confidence'] + 0.3*kw_s + 0.2*mf_s), 3)
    rat    = (f'Docs={fs["doc_count"]} | KW={len(fs["matched_keywords"])}/{tot_kw} ({kw_s*100:.0f}%) | '
              f'MF={len(fs["matched_mandatory_fields"])}/{tot_mf} ({mf_s*100:.0f}%) | Raw={raw:.2f}=>snap={final}')
    return {'raw_score':round(raw,2),'final_score':final,'confidence':conf,
            'review_flag':conf<CONFIDENCE_THRESHOLD,'rationale':rat}

pre_eval = evaluate_criteria(fact_sheet, kb_data)
print('Pre-LLM Criterion Evaluation:')
print(f'  Raw score   : {pre_eval["raw_score"]}')
print(f'  Final score : {pre_eval["final_score"]}')
print(f'  Confidence  : {pre_eval["confidence"]}')
print(f'  Review flag : {pre_eval["review_flag"]}')
print(f'  Rationale   : {pre_eval["rationale"]}')

## 11. VLLM Juror + SR-Aware Consensus

In [ ]:
def call_juror(fs: Dict, kb: Dict, role: str = 'compliance') -> Dict:
    sr_str = json.dumps(kb.get('scoring_rules',{}), indent=2, ensure_ascii=False)
    ev_str = json.dumps(fs['top_evidence'], indent=2, ensure_ascii=False)
    mf_met = len(fs['matched_mandatory_fields'])
    mf_tot = mf_met + len(fs['missing_mandatory_fields'])
    system = (
        f'You are a CSMS {role.title()} Juror for the Maritime industry. '
        'Evaluate category compliance strictly based on the scoring rules. '
        'Return ONLY valid JSON with keys: score (0/3/6/10), confidence (0.0-1.0), reasoning.'
    )
    user = (
        f'CATEGORY: {fs["category_name"]}\n\n'
        f'SCORING RULES (follow the IF-THEN decision logic exactly):\n{sr_str}\n\n'
        f'SUBMITTED EVIDENCE:\n{ev_str}\n\n'
        f'MANDATORY FIELDS MET: {mf_met}/{mf_tot}\n'
        f'MISSING MANDATORY FIELDS: {fs["missing_mandatory_fields"]}\n\n'
        f'Reasoning hint: {kb.get("scoring_rules",{}).get("reasoning_hint","")}\n\n'
        'Evaluate step by step and return your JSON verdict.'
    )
    try:
        resp = vllm_client.chat.completions.create(
            model=VLLM_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, max_tokens=1024,
        )
        raw = resp.choices[0].message.content.strip()
        print(f"  [DEBUG] LLM Raw Output:\n{raw}\n")
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1:
            raw = raw[start:end+1]
        res = json.loads(raw)
        return {'score':snap(res.get('score',6)),'confidence':float(res.get('confidence',0.7)),
                'reasoning':res.get('reasoning',''),'error':None}
    except Exception as e:
        return {'score':pre_eval['final_score'],'confidence':pre_eval['confidence'],
                'reasoning':f'LLM failed: {e}','error':str(e)}

def apply_gates(llm_score: int, fs: Dict) -> int:
    if not fs['has_documents']:
        return 0
    if llm_score == 10 and len(fs['missing_mandatory_fields']) > 0:
        print(f'  Gate: 10->6 ({len(fs["missing_mandatory_fields"])} mandatory fields missing)')
        return 6
    if llm_score == 0 and fs['has_documents']:
        print('  Gate: 0->3 (documents exist)')
        return 3
    return llm_score

print('Calling VLLM Compliance Juror...')
t0 = time.time()
juror_result = call_juror(fact_sheet, kb_data, role='compliance')
print(f'Juror response in {time.time()-t0:.1f}s:')
print(f'  Raw juror score : {juror_result["score"]}')
print(f'  Confidence      : {juror_result["confidence"]}')
if juror_result['error']:
    print(f'  ERROR           : {juror_result["error"]}')
print(f'  Reasoning       :')
print(juror_result['reasoning'])

consensus_score = apply_gates(juror_result['score'], fact_sheet)
final_confidence = juror_result['confidence']
review_flag = final_confidence < CONFIDENCE_THRESHOLD

print()
print('='*60)
print(f'FINAL CONSENSUS SCORE : {consensus_score}')
print(f'Confidence            : {final_confidence:.3f}')
print(f'Review flag           : {review_flag}')
print('='*60)

## 12. Final Result & Audit Report

In [ ]:
result = {
    'category_id':     TARGET_CATEGORY_ID,
    'category_name':   fact_sheet['category_name'],
    'expected_score':  expected_score,
    'ai_score':        consensus_score,
    'pre_llm_score':   pre_eval['final_score'],
    'raw_score':       pre_eval['raw_score'],
    'confidence':      round(final_confidence, 3),
    'review_flag':     review_flag,
    'is_correct':      (consensus_score == expected_score) if expected_score is not None else None,
    'juror_reasoning': juror_result['reasoning'],
    'validation': {
        'doc_count':      validation['doc_count'],
        'keyword_score':  validation['keyword_score'],
        'field_score':    validation['field_score'],
        'matched_kw':     validation['matched_keywords'],
        'missing_kw':     validation['missing_keywords'],
        'matched_mf':     validation['matched_mandatory_fields'],
        'missing_mf':     validation['missing_mandatory_fields'],
    }
}

icon  = 'CORRECT' if result['is_correct'] else ('WRONG' if result['is_correct'] is False else 'N/A')
bar_n = int(final_confidence * 20)
bar   = 'X'*bar_n + '.'*(20-bar_n)

def pct(n, d): return f'{n/d*100:.0f}%' if d else 'N/A'

html = f'''
<div style='font-family:Segoe UI,sans-serif;max-width:820px;background:#0f172a;color:#e2e8f0;border-radius:12px;padding:24px;margin:8px 0'>
  <h2 style='color:#38bdf8;margin:0 0 2px 0'>CSMS Category {TARGET_CATEGORY_ID} Test Result</h2>
  <p style='color:#64748b;margin:0 0 20px 0'>{fact_sheet['category_name']}</p>
  <div style='display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;margin-bottom:16px'>
    <div style='background:#1e293b;border-radius:8px;padding:14px;text-align:center'>
      <div style='font-size:2.4em;font-weight:800;color:#34d399'>{result['ai_score']}</div>
      <div style='color:#64748b;font-size:0.8em'>AI Score</div>
    </div>
    <div style='background:#1e293b;border-radius:8px;padding:14px;text-align:center'>
      <div style='font-size:2.4em;font-weight:800;color:#fbbf24'>{result['expected_score'] or '?'}</div>
      <div style='color:#64748b;font-size:0.8em'>Expected</div>
    </div>
    <div style='background:#1e293b;border-radius:8px;padding:14px;text-align:center'>
      <div style='font-size:2.4em;font-weight:800;color:#{'34d399' if result['is_correct'] else 'f87171'}'>{icon}</div>
      <div style='color:#64748b;font-size:0.8em'>Match</div>
    </div>
  </div>
  <div style='background:#1e293b;border-radius:8px;padding:14px;margin-bottom:10px'>
    <span style='color:#94a3b8;font-size:0.85em'>Confidence [{bar}] {final_confidence:.1%}</span><br>
    <span style='color:#94a3b8;font-size:0.8em'>Pre-LLM: {result['pre_llm_score']} | Raw: {result['raw_score']}</span>
  </div>
  <div style='background:#1e293b;border-radius:8px;padding:14px;margin-bottom:10px'>
    <div style='color:#38bdf8;font-weight:600;margin-bottom:6px'>Validation</div>
    Docs: {validation['doc_count']} &nbsp;|&nbsp;
    Keywords: {pct(len(validation['matched_keywords']),len(validation['matched_keywords'])+len(validation['missing_keywords']))} &nbsp;|&nbsp;
    Fields: {pct(len(validation['matched_mandatory_fields']),len(validation['matched_mandatory_fields'])+len(validation['missing_mandatory_fields']))}<br>
    <span style='color:#64748b;font-size:0.8em'>Missing fields: {', '.join(validation['missing_mandatory_fields'][:4]) or 'None'}</span>
  </div>
  <div style='background:#1e293b;border-radius:8px;padding:14px'>
    <div style='color:#38bdf8;font-weight:600;margin-bottom:6px'>Juror Reasoning</div>
    <div style='color:#cbd5e1;font-size:0.88em;white-space:pre-wrap'>{juror_result['reasoning'][:800]}</div>
  </div>
</div>'''
display(HTML(html))

print('JSON Result:')
print(json.dumps({k:v for k,v in result.items() if k != 'validation'}, ensure_ascii=False, indent=2))

out = os.path.join(OUTPUT_DIR, 'reports', f'single_test_{TARGET_CATEGORY_ID}.json')
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f'\nSaved to {out}')
conn.close()